# Causal Tracing (Alt-style)

Standalone end-to-end causal mediation analysis implementing the approach
described at [alt tracing workflow](https://alt tracing workflow).

**Pipeline:**
1. Corrupt subject embeddings with Gaussian noise (scale 3.0 × embedding std)
2. Restore each layer one at a time at the last subject token position
3. Average over 10 independent runs with fresh noise per run
4. Rank layers by probability mass recovered for the target token
5. Fall back to middle third when the signal is noisy

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for p in [ROOT, *ROOT.parents]:
    if (p / 'src' / 'main.py').exists():
        ROOT = p
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')

In [ ]:
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

from src.handlers.rome import ModelHandler
from src.utils import load_dataset
from src.causal_trace.causal_trace import filter_dataset, preprocess_prompt
from src.causal_trace.alt_trace import (
    trace_prompt,
    select_layers,
    _ensure_noise_multiplier,
    TraceResult,
    LayerSelection,
)

print('Imports OK')

## 2. Load model via Hydra config

Set `MODEL_CONFIG` to any model config name from `src/config/model/`.

In [ ]:
MODEL_CONFIG = 'gpt2-xl'

from hydra import initialize, compose
from hydra.core.config_store import ConfigStore

config_dir = str(ROOT / 'src' / 'config')
with initialize(version_base=None, config_path=None):
    cfg = compose(
        config_name='config',
        overrides=[
            f'model={MODEL_CONFIG}',
            'command=alt-trace',
        ],
    )

print(f'Model: {cfg.model.name}')
print(f'Layers: {cfg.model.get("layer", "N/A")}')
print(f'Noise multiplier: {cfg.model.get("corruption_noise_multiplier", "auto")}')

In [ ]:
handler = ModelHandler(cfg)
print(f'Loaded {cfg.model.name} — {handler.num_of_layers} layers')

dataset = load_dataset(cfg)
df_dataset = filter_dataset(dataset['requested_rewrite'])
print(f'Dataset: {len(df_dataset)} prompts available')

_ensure_noise_multiplier(handler, cfg, df_dataset)
print(f'Noise multiplier: {handler._noise_multiplier:.6f}')

## 3. Run causal trace on sample prompts

Traces `NUM_PROMPTS` prompts with `NUM_RUNS` independent noise samples each.
Restoration is applied only at the **last subject token** position.

In [ ]:
NUM_PROMPTS = 5
NUM_RUNS = 10

results = []
total = 0
failed = 0

for prompt_dict in df_dataset.itertuples():
    if total - failed >= NUM_PROMPTS:
        break
    total += 1

    preprocessed = preprocess_prompt(handler, prompt_dict)
    if preprocessed is None:
        failed += 1
        continue

    prompt_ids, subject_positions = preprocessed
    result = trace_prompt(
        handler,
        prompt_ids,
        subject_positions,
        prompt_dict.target_true['str'],
        num_runs=NUM_RUNS,
    )

    if result is None:
        target_str = prompt_dict.target_true['str']
        print(f'  SKIP (wrong clean token): "{prompt_dict.subject}" -> expected "{target_str}"')
        failed += 1
        continue

    result.prompt_idx = prompt_dict.Index
    result.subject = prompt_dict.subject
    results.append(result)
    print(f'  OK: "{prompt_dict.subject}" → "{result.target}"  '
          f'clean={result.clean_prob:.4f}  corrupt={result.corrupt_prob:.4f}  '
          f'peak=L{int(np.argmax(result.per_layer_probs))}')

print(f'\nDone: {len(results)} successful, {failed} failed, {total} total')

## 4. Plot per-layer restoration curves

In [ ]:
num_layers = handler.num_of_layers
mid_start = num_layers // 3
mid_end = 2 * num_layers // 3

fig, axes = plt.subplots(1, min(len(results), 5), figsize=(5 * min(len(results), 5), 4), sharey=False)
if len(results) == 1:
    axes = [axes]

for i, r in enumerate(results[:5]):
    ax = axes[i]
    layers = np.arange(num_layers)
    probs = r.per_layer_probs

    ax.axvspan(mid_start, mid_end, alpha=0.15, color='orange', label='middle third')
    ax.bar(layers, probs, width=0.8, color='steelblue', edgecolor='none')

    peak = int(np.argmax(probs))
    ax.bar(peak, probs[peak], width=0.8, color='crimson', edgecolor='none', label=f'peak L{peak}')

    ax.set_title(f'{r.subject} → {r.target}', fontsize=9)
    ax.set_xlabel('Layer')
    if i == 0:
        ax.set_ylabel('Restoration prob')
    ax.set_xlim(-0.5, num_layers - 0.5)

fig.suptitle(f'Causal trace — {cfg.model.name} ({NUM_RUNS} runs/prompt)', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Aggregate and select layers

Average restoration probabilities across all traced prompts, then rank.
When the signal is noisy the candidate pool falls back to the middle third.

In [ ]:
avg_probs = np.mean([r.per_layer_probs for r in results], axis=0)
selection = select_layers(avg_probs, num_layers)

print(selection.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
layers = np.arange(num_layers)

ax.axvspan(mid_start, mid_end, alpha=0.12, color='orange', label='middle third')
ax.bar(layers, avg_probs, width=0.8, color='steelblue', edgecolor='none', label='avg restoration prob')

peak = int(np.argmax(avg_probs))
best = selection.best_layer
ax.bar(peak, avg_probs[peak], width=0.8, color='crimson', edgecolor='none', label=f'peak L{peak}')
if best != peak:
    ax.bar(best, avg_probs[best], width=0.8, color='darkgreen', edgecolor='none', label=f'selected L{best}')

ax.set_title(
    f'{cfg.model.name} — {len(results)} prompts × {NUM_RUNS} runs  '
    f'    (quality: {selection.signal_quality}, fallback: {selection.used_middle_third_fallback})',
    fontsize=11,
)
ax.set_xlabel('Layer')
ax.set_ylabel('Avg restoration probability')
ax.set_xlim(-0.5, num_layers - 0.5)
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Candidate table

In [ ]:
print(f"{'Rank':<6} {'Layer':<8} {'Restoration Prob':<20} {'Middle Third':<14}")
print('-' * 48)
for c in selection.candidates[:10]:
    tag = '✓' if c.in_middle_third else ''
    print(f'{c.rank:<6} L{c.layer:<7} {c.restoration_prob:<20.6f} {tag:<14}')

print(f'\nBest layer: L{selection.best_layer}')
print(f'Signal quality: {selection.signal_quality}')
print(f'Middle-third fallback: {selection.used_middle_third_fallback}')

## 7. Save results (optional)

In [ ]:
SAVE = False

if SAVE and results:
    from src.causal_trace.alt_trace import _save_results

    csv_path, json_path = _save_results(results, avg_probs, selection, cfg)
    print(f'CSV:  {csv_path}')
    print(f'JSON: {json_path}')
else:
    print('Set SAVE = True to write CSV + JSON to analysis_out/')